In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import plotly.io as pio

class MentalHealthPredictor:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.feature_names = ['age', 'stress', 'sleep', 'academic_performance']
        self.history = []

    def upload_files(self):
        print("Please upload your dataset files (CSV format)...")
        uploaded = files.upload()
        clear_output()
        return list(uploaded.keys())

    def load_data(self, file_paths):
        dataframes = []

        for file_path in file_paths:
            try:
                df = pd.read_csv(file_path)
                processed_df = pd.DataFrame()
                df.columns = df.columns.str.lower().str.strip()

                if 'academic pressure' in df.columns:
                    processed_df['age'] = pd.to_numeric(df['age'], errors='coerce')
                    processed_df['stress'] = df['academic pressure']
                    processed_df['sleep'] = pd.to_numeric(df['sleep duration'].map({
                        'Less than 5 hours': 4, '5-6 hours': 5.5,
                        '7-8 hours': 7.5, 'More than 8 hours': 9
                    }), errors='coerce')
                    processed_df['academic_performance'] = pd.to_numeric(df['cgpa'], errors='coerce')
                    processed_df['depression'] = df['depression'].astype(int)
                    dataframes.append(processed_df)

                elif 'studystresslevel' in df.columns:
                    processed_df['age'] = pd.to_numeric(df['age'], errors='coerce')
                    processed_df['stress'] = df['studystresslevel']
                    processed_df['sleep'] = df['sleepquality']
                    processed_df['academic_performance'] = pd.to_numeric(df['cgpa'], errors='coerce')
                    processed_df['depression'] = df['depression'].astype(int)
                    dataframes.append(processed_df)

                elif 'anxiety value' in df.columns:
                    processed_df['age'] = df['1. age'].map({
                        'Below 18': 17, '18-22': 20,
                        '23-26': 24, '27-30': 28
                    })
                    processed_df['stress'] = df['stress value']
                    processed_df['academic_performance'] = df['6. current cgpa'].map({
                        'Below 2.50': 2.25, '2.50 - 2.99': 2.75,
                        '3.00 - 3.39': 3.20, '3.40 - 3.79': 3.60,
                        '3.80 - 4.00': 3.90
                    })
                    processed_df['sleep'] = 6
                    processed_df['depression'] = (df['depression value'] > 15).astype(int)
                    dataframes.append(processed_df)

                print(f"Successfully processed: {file_path}")
            except Exception as e:
                print(f"Error processing {file_path}: {str(e)}")

        if not dataframes:
            raise ValueError("No valid data was processed from the input files")

        self.data = pd.concat(dataframes, ignore_index=True)
        self.data = self.data.fillna(self.data.median())
        print(f"\nTotal records processed: {len(self.data)}")
        return self.data

    def preprocess_data(self):
        X = self.data[self.feature_names]
        y = self.data['depression']
        X_scaled = self.scaler.fit_transform(X)
        return train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

    def train_model(self, X_train, y_train):
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
        self.model = VotingClassifier(
            estimators=[('rf', rf), ('gb', gb)],
            voting='soft'
        )
        self.model.fit(X_train, y_train)

    def evaluate_model(self, X_test, y_test):
        y_pred = self.model.predict(X_test)
        accuracy = np.mean(y_pred == y_test)
        print(f"\nModel Accuracy: {accuracy:.2f}")
        self.visualize_feature_importance()
        return accuracy

    def create_prediction_interface(self):
        age_input = widgets.IntSlider(
            min=16, max=45, value=20,
            description='Age:',
            style={'description_width': 'initial'}
        )

        stress_input = widgets.IntSlider(
            min=1, max=10, value=5,
            description='Stress Level (1-10):',
            style={'description_width': 'initial'}
        )

        sleep_input = widgets.FloatSlider(
            min=2, max=10, value=7,
            description='Sleep Hours:',
            style={'description_width': 'initial'}
        )

        academic_input = widgets.FloatSlider(
            min=1, max=10, value=3,
            step=0.1,
            description='CGPA (1-10):',
            style={'description_width': 'initial'}
        )

        predict_button = widgets.Button(
            description='Predict Mental Health Status',
            style={'button_color': '#4CAF50'}
        )

        output = widgets.Output()

        def on_button_click(b):
            with output:
                clear_output()
                prediction = self.predict_depression_risk(
                    age_input.value,
                    stress_input.value,
                    sleep_input.value,
                    academic_input.value
                )

                # self.plot_radar_chart(age_input.value, stress_input.value,
                #                     sleep_input.value, academic_input.value)
                self.plot_risk_gauge(prediction['probability'])
                self.display_recommendations(prediction)
                self.update_history(age_input.value, stress_input.value,
                                  sleep_input.value, academic_input.value,
                                  prediction)
                self.plot_history_trend()

        predict_button.on_click(on_button_click)

        print("Enter your details for mental health prediction:")
        display(widgets.VBox([
            age_input, stress_input, sleep_input,
            academic_input, predict_button, output
        ]))

    # def plot_radar_chart(self, age, stress, sleep, academic):
    #     categories = ['Age', 'Stress', 'Sleep', 'Academic']
    #     values = [age/45, stress/10, sleep/10, academic/10]

    #     fig = go.Figure()
    #     fig.add_trace(go.Scatterpolar(
    #         r=values,
    #         theta=categories,
    #         fill='toself',
    #         name='Your Factors'
    #     ))

    #     fig.update_layout(
    #         polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    #         showlegend=True,
    #         title='Your Mental Health Factors'
    #     )
    #     fig.show()

    def plot_risk_gauge(self, risk_probability):
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=risk_probability * 100,
            domain={'x': [0, 1], 'y': [0, 1]},
            title={'text': "Depression Risk Level"},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': "darkblue"},
                'steps': [
                    {'range': [0, 30], 'color': "lightgreen"},
                    {'range': [30, 70], 'color': "yellow"},
                    {'range': [70, 100], 'color': "red"}
                ],
                'threshold': {
                    'line': {'color': "red", 'width': 4},
                    'thickness': 0.75,
                    'value': 70
                }
            }
        ))
        fig.show()

    def display_recommendations(self, prediction):
        recommendations = {
            'High': [
                "- Urgent: Seek professional help",
                "- Contact student counseling services",
                "- Discuss academic adjustments",
                "- Establish regular sleep schedule",
                "- Join support groups"
            ],
            'Medium': [
                "- Regular mental health check-ins",
                "- Practice stress management",
                "- Maintain study-life balance",
                "- Consider counseling services"
            ],
            'Low': [
                "- Continue healthy habits",
                "- Regular exercise",
                "- Maintain social connections",
                "- Practice self-care"
            ]
        }

        html_content = self.generate_html_report(prediction, recommendations)
        display(HTML(html_content))

    def generate_html_report(self, prediction, recommendations):
        return f"""
        <div style='background-color: #28282B ; padding: 20px; border-radius: 10px;'>
            <h2 style='color: #FFFFFF;'>Mental Health Assessment Results</h2>
            <p style='font-size: 18px;'>Risk Level:
                <span style='color: {'red' if prediction['risk_level'] == 'High'
                                   else 'orange' if prediction['risk_level'] == 'Medium'
                                   else 'green'};'>
                    {prediction['risk_level']}
                </span>
            </p>
            <p style='font-size: 18px;'>Depression Probability: {prediction['probability']:.2f}</p>

            <h3 style='color: #FFFFFF;'>Recommendations:</h3>
            <ul style='list-style-type: none;'>
                {''.join(f"<li style='margin: 10px 0;'>{rec}</li>"
                        for rec in recommendations[prediction['risk_level']])}
            </ul>

            <h3 style='color: #FFFFFF;'>Next Steps:</h3>
            <ol style='color: #FFFFFF;'>
                <li>Monitor mental health regularly</li>
                <li>Set realistic academic goals</li>
                <li>Build support network</li>
                <li>Practice stress management</li>
            </ol>
        </div>
        """

    def update_history(self, age, stress, sleep, academic, prediction):
        self.history.append({
            'timestamp': datetime.now(),
            'age': age,
            'stress': stress,
            'sleep': sleep,
            'academic': academic,
            'risk_level': prediction['risk_level'],
            'probability': prediction['probability']
        })

    # def plot_history_trend(self):
    #     if len(self.history) > 1:
    #         df = pd.DataFrame(self.history)
    #         fig = go.Figure()

    #         fig.add_trace(go.Scatter(
    #             x=df['timestamp'],
    #             y=df['probability'],
    #             mode='lines+markers',
    #             name='Risk Probability'
    #         ))

    #         fig.update_layout(
    #             title='Mental Health Risk Trend',
    #             xaxis_title='Time',
    #             yaxis_title='Risk Probability',
    #             template='plotly_white'
    #         )

    #         fig.show()

    def visualize_feature_importance(self):
        rf_model = self.model.named_estimators_['rf']
        importances = rf_model.feature_importances_

        fig = go.Figure([go.Bar(
            x=self.feature_names,
            y=importances,
            marker_color=['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']
        )])

        fig.update_layout(
            title='Feature Importance in Depression Prediction',
            xaxis_title='Features',
            yaxis_title='Importance',
            template='plotly_dark',
            bargap=0.2,
            height=500
        )

        fig.show()

    def predict_depression_risk(self, age, stress_level, sleep_hours, academic_performance):
        input_data = np.array([[age, stress_level, sleep_hours, academic_performance]])
        input_scaled = self.scaler.transform(input_data)
        prediction = self.model.predict_proba(input_scaled)[0]

        return {
            'risk_level': 'High' if prediction[1] > 0.7
                         else 'Medium' if prediction[1] > 0.3
                         else 'Low',
            'probability': prediction[1]
        }

def main():
    predictor = MentalHealthPredictor()
    uploaded_files = predictor.upload_files()
    predictor.load_data(uploaded_files)
    X_train, X_test, y_train, y_test = predictor.preprocess_data()
    predictor.train_model(X_train, y_train)
    predictor.evaluate_model(X_test, y_test)
    print("\nInteractive Prediction Interface:")
    predictor.create_prediction_interface()

if __name__ == "__main__":
    main()

Successfully processed: Raw Data.csv
Successfully processed: Student Mental health.csv
Successfully processed: MentalHealthSurvey.csv
Successfully processed: Student Depression Dataset.csv
Successfully processed: mentalhealth_dataset.csv

Total records processed: 30878

Model Accuracy: 0.71



Interactive Prediction Interface:
Enter your details for mental health prediction:
